# 🧠 Modelos Não Supervisionados de Machine Learning

**MBA em Data Science e Analytics (USP/ESALQ)**  
**Prof. Dr. Wilson Tarantin Junior**

---

Este notebook apresenta, de forma didática, as três grandes técnicas de aprendizado não supervisionado estudadas no curso:

| # | Técnica | Tipo de Dado | Objetivo Principal |
|---|---------|-------------|--------------------|
| 1 | **Análise de Cluster (K-Means)** | Quantitativo | Agrupar observações semelhantes |
| 2 | **Análise Fatorial (PCA)** | Quantitativo | Reduzir dimensões e extrair fatores latentes |
| 3 | **Análise de Correspondência (ANACOR/ACM)** | Qualitativo | Mapear associações entre categorias |

> **O que é aprendizado não supervisionado?**  
> São técnicas que encontram **estruturas ocultas** nos dados sem usar uma variável resposta (rótulo).  
> O algoritmo aprende padrões apenas a partir das próprias características das observações.

---
# 📍 Parte 1 — Análise de Cluster (K-Means)

## 🧩 O que é a Análise de Cluster?

- Técnica de **segmentação** que agrupa observações com base em **similaridade**
- Não há variável resposta: os grupos emergem naturalmente da estrutura dos dados
- Amplamente usada em: segmentação de clientes, análise de mercado, bioinformática, reconhecimento de padrões

> **Intuição:** imagine espalhar pontos num plano — a clusterização identifica automaticamente quais pontos “moram perto uns dos outros” e os agrupa.

## ⚙️ Como o K-Means funciona?

O algoritmo K-Means é iterativo e segue 4 passos:

**Passo 1 — Inicialização**
- Define-se **K** (número de clusters desejado)
- São escolhidos K pontos aleatórios como **centróides iniciais**

**Passo 2 — Atribuição**
- Cada observação é atribuída ao centróide **mais próximo** (distância euclidiana)

**Passo 3 — Atualização**
- O centróide de cada cluster é recalculado como a **média** das observações atribuídas

**Passo 4 — Convergência**
- Os passos 2 e 3 se repetem até que as atribuições não mudem mais (convergência)

> O objetivo do K-Means é **minimizar o WCSS** (Within-Cluster Sum of Squares):  
> $$WCSS = \sum_{k=1}^{K} \sum_{i \in C_k} ||x_i - \mu_k||^2$$  
> onde $\mu_k$ é o centróide do cluster $k$ e $x_i$ são as observações pertencentes a ele.

## 📏 Padronização — Por que é obrigatória?

O K-Means usa **distâncias euclidianas**. Variáveis com escalas diferentes produzem distâncias enviesadas.

| Variável | Escala | Sem Padronização |
|----------|--------|------------------|
| Renda (R\$) | 0 – 50.000 | **Domina** o cálculo |
| Idade (anos) | 18 – 80 | Pouco impacto |
| Nº filhos | 0 – 6 | Quase sem impacto |

**Solução: Z-Score**

$$z = \frac{x - \bar{x}}{s}$$

- Transforma cada variável para **média 0** e **desvio-padrão 1**
- Todas as variáveis passam a contribuir igualmente para a distância

## 📉 Como escolher o K? Método Elbow

Não existe um K “verdadeiro” — precisamos de critérios para orientá-lo.

**Lógica do Elbow:**
- Testamos K de 1 a 10 (ou mais)
- Para cada K, registramos o WCSS
- O gráfico forma uma curva decrescente: o **cotovelo** (ponto de inflexão) indica onde o ganho de qualidade começa a ser pequeno

> **Regra prática:** o K ótimo está no ponto onde adicionar mais um cluster traz redução marginal do WCSS cada vez menor.

## 🔮 Como escolher o K? Método da Silhueta

O **coeficiente de silhueta** mede, para cada observação, quão bem ela está alocada no seu cluster.

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}$$

- **$a(i)$** = distância média da observação $i$ para as demais **no mesmo cluster** (coesão)
- **$b(i)$** = menor distância média da observação $i$ para qualquer **outro cluster** (separação)

| Valor de $s(i)$ | Interpretação |
|-----------------|----------------|
| Próximo de **+1** | Bem alocado, cluster coeso e separado |
| Próximo de **0** | Na fronteira entre dois clusters |
| Próximo de **−1** | Provavelmente mal alocado |

> O K ótimo pela silhueta é aquele que **maximiza a silhueta média** de todas as observações.  
> **Recomendação:** use Elbow + Silhueta em conjunto para uma decisão mais robusta.

## 🧪 Validação com ANOVA

Após definir os clusters, usamos a **ANOVA (Análise de Variância)** para confirmar que as variáveis usadas distinguem significativamente os grupos.

- **H₀:** as médias são iguais entre todos os clusters
- **H₁:** pelo menos um par de clusters tem médias diferentes
- Critério: **p-valor < 0,05** → rejeita H₀ → a variável discrimina os clusters

> **η² (Eta-quadrado):** mede o **tamanho do efeito** — a proporção da variância da variável explicada pelo pertencimento ao cluster.  
> $\eta^2 = SS_{entre} / SS_{total}$  
> Valores **> 0,14** indicam efeito grande (grande poder discriminatório da variável).

## ✅ Quando usar a Análise de Cluster?

| Situação | Adequado? |
|-----------|----------|
| Variáveis quantitativas com escalas diferentes | ✅ Sim (com Z-Score) |
| Deseja segmentar clientes, países, produtos | ✅ Sim |
| Número de grupos desconhecido a priori | ✅ Sim (usa Elbow + Silhueta) |
| Apenas variáveis qualitativas | ❌ Não (use ACM) |
| Objetivo é prever uma variável resposta | ❌ Não (use classificação supervisionada) |

---
# 📐 Parte 2 — Análise Fatorial por Componentes Principais (PCA)

## 🧩 O que é a Análise Fatorial?

- Técnica de **redução de dimensionalidade** que transforma um conjunto de variáveis correlacionadas em um conjunto menor de **fatores latentes** não correlacionados
- Cada fator captura uma **dimensão subjacente** que explica parte da variância das variáveis originais
- Usada para: criar índices compostos, eliminar multicolinearidade, ranking, análise exploratória

> **Intuição:** se você tem 10 variáveis muito correlacionadas entre si, provavelmente existe um fenômeno latente comum que as explica. A PCA encontra e quantifica esse fenômeno.

## 📊 O Modelo Fatorial

Cada variável observada $X_j$ é expressa como combinação linear dos fatores:

$$X_j = \lambda_{j1}F_1 + \lambda_{j2}F_2 + \cdots + \lambda_{jm}F_m + \varepsilon_j$$

- **$\lambda_{jk}$** = carga fatorial da variável $j$ no fator $k$ (correlação entre variável e fator)
- **$F_k$** = fator latente (não observável diretamente)
- **$\varepsilon_j$** = parte da variável não explicada pelos fatores (especificidade)

> **Comunalidade** ($h^2_j$): proporção da variância de $X_j$ explicada pelos fatores retidos.  
> $h^2_j = \lambda_{j1}^2 + \lambda_{j2}^2 + \cdots + \lambda_{jm}^2$  
> Quanto mais próxima de 1, melhor a variável é representada pelo modelo fatorial.

## 🧪 Pré-requisito: Teste de Esfericidade de Bartlett

Antes de aplicar a PCA, verificamos se as variáveis são suficientemente correlacionadas.

- **H₀:** a matriz de correlações é uma **matriz identidade** (variáveis completamente independentes)
- **H₁:** existem correlações significativas entre as variáveis
- Critério: **p-valor < 0,05** → rejeita H₀ → PCA é adequada

> Se a matriz de correlações fosse uma identidade, cada variável seria completamente independente das demais.  
> Nesse cenário, não haveria fator latente a ser extraído — cada variável já seria seu próprio “fator”.

## 🔢 Autovalores e o Critério da Raiz Latente

A PCA decompõe a matriz de correlações em **autovalores** ($\lambda$) e **autovetores**.

- Cada autovalor mede a **variância total explicada** por aquele fator
- A soma de todos os autovalores = número de variáveis ($p$)

**Critério da Raiz Latente (Kaiser):**

$$\text{Reter fator } k \iff \lambda_k > 1$$

| Por quê autovalor > 1? | Explicação |
|------------------------|------------|
| Cada variável padronizada tem variância = 1 | Fator deve explicar mais que uma variável individual |
| Autovalor = 1 → fator equivale a 1 variável | Abaixo de 1 não há compressão de informação |

> **Critério alternativo:** reter fatores até atingir **60–70% de variância acumulada** explicada.

## 🔗 Cargas Fatoriais — Interpretando os Fatores

As **cargas fatoriais** ($\lambda_{jk}$) são as correlações entre cada variável e cada fator.

| Carga (valor absoluto) | Interpretação |
|-----------------------|----------------|
| **> 0,70** | Associação muito forte |
| **0,50 – 0,70** | Associação forte (limiar usual de relevância) |
| **0,30 – 0,50** | Associação moderada |
| **< 0,30** | Associação fraca — variável pouco ligada ao fator |

> Um fator é **interpretado** pelo conjunto de variáveis com cargas absolutas **> 0,5** nele.  
> O analista **nomeia** o fator com base no padrão semântico dessas variáveis.  
> Ex.: cargas altas de `renda`, `escolaridade` e `patrimônio` → fator pode ser chamado de **“Nível Socioeconômico”**.

## 🏋️ Scores Fatoriais — Criando o Índice

Os **scores fatoriais** são os coeficientes que ponderam cada variável para calcular o escore de uma observação:

$$F_k = w_{k1}z_1 + w_{k2}z_2 + \cdots + w_{kp}z_p$$

- **$w_{kj}$** = peso (score) da variável $j$ no fator $k$
- **$z_j$** = valor padronizado da variável $j$

> O escore fatorial de cada observação é o valor que essa observação recebe no fator latente.  
> Pode ser usado como **ranking** (ex.: desempenho educacional de países no PISA),  
> como **feature** em modelos preditivos, ou como base para **segmentação**.

---
**Diferença entre cargas e scores:**

| | Cargas Fatoriais | Scores Fatoriais |
|-|-----------------|------------------|
| **O quê mede** | Correlação variável-fator | Coeficiente de ponderação |
| **Escala** | −1 a +1 | Qualquer valor real |
| **Para quê serve** | Interpretar o fator | Calcular o escore de cada observação |

## ✅ Quando usar a Análise Fatorial (PCA)?

| Situação | Adequado? |
|-----------|----------|
| Variáveis quantitativas altamente correlacionadas | ✅ Sim |
| Criar um índice composto ou ranking | ✅ Sim |
| Eliminar multicolinearidade antes de regressão | ✅ Sim |
| Variáveis independentes entre si (sem correlação) | ❌ Não (Bartlett falhará) |
| Variáveis qualitativas | ❌ Não (use ACM) |

---
# 🗺️ Parte 3 — Análise de Correspondência (ANACOR / ACM)

## 🧩 O que é a Análise de Correspondência?

- Técnica de **redução de dimensionalidade para dados categóricos** que representa graficamente as associações entre categorias de variáveis qualitativas
- Existe em duas versões:

| Versão | Sigla | Quando usar |
|--------|-------|-------------|
| Análise de Correspondência Simples | **ANACOR** | 2 variáveis qualitativas |
| Análise de Correspondência Múltipla | **ACM** | 3 ou mais variáveis qualitativas |

> **Intuição:** assim como a PCA projeta variáveis numéricas em eixos de máxima variância,  
> a ANACOR/ACM projeta **categorias** em eixos de máxima **inércia** (variação das associações).

## 🧪 Pré-requisito: Teste Qui-Quadrado ($\chi^2$)

A Análise de Correspondência só faz sentido se as variáveis **não são independentes**.

- **H₀:** as variáveis são independentes (sem associação)
- **H₁:** existe associação entre as variáveis
- Critério: **p-valor < 0,05** → rejeita H₀ → há associação → ANACOR/ACM é válida

$$\chi^2 = \sum_{i}\sum_{j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

- $O_{ij}$ = frequência **observada** na célula $(i,j)$
- $E_{ij}$ = frequência **esperada** sob independência: $E_{ij} = \frac{n_{i\cdot} \cdot n_{\cdot j}}{n}$

## 📊 Inércia — A “Variância” das Associações

Na Análise de Correspondência, a **inércia** é o equivalente à variância explicada na PCA.

$$\text{Inércia Total} = \frac{\chi^2}{n}$$

- Mede o **grau de afastamento** da independência entre as categorias
- Maior inércia = maior diversidade nas associações entre categorias

**Dimensões da ANACOR:**

$$m = \min(I-1, J-1)$$

- $I$ = número de categorias da variável linha
- $J$ = número de categorias da variável coluna
- Com 2 categorias em uma das variáveis ($I=2$ ou $J=2$) → apenas **1 dimensão** → não é possível mapa bidimensional

## 📍 Coordenadas-Padrão e o Mapa Perceptual

As **coordenadas-padrão** posicionam as categorias no espaço dimensional da correspondência.

Na ACM, obtidas por:

$$\text{Coordenada-Padrão} = \frac{\text{Coordenada Principal}}{\sqrt{\lambda_k}}$$

**Como ler o mapa perceptual:**

| Situação no Mapa | Interpretação |
|-----------------|----------------|
| Categorias **próximas** | Tendem a ocorrer juntas nas mesmas observações |
| Categorias **distantes** | Tendem a ocorrer em observações diferentes |
| Categoria **próxima ao centro** | Comportamento próximo da média (pouca informação) |
| Categoria **distante do centro** | Comportamento atípico, perfil bem definido |

> O **eixo horizontal** (Dimensão 1) captura a maior parte da inércia.  
> O **eixo vertical** (Dimensão 2) captura a segunda maior parte.  
> A soma dessas duas proporções indica quanto da estrutura de associações o mapa 2D representa.

## 📉 Resíduos Padronizados Ajustados

Quando não é possível gerar um mapa bidimensional (ex.: variável binária), a análise termina nos **resíduos padronizados ajustados**.

$$r_{ij} = \frac{O_{ij} - E_{ij}}{\sqrt{E_{ij} \cdot (1 - p_{i\cdot}) \cdot (1 - p_{\cdot j})}}$$

| Valor do Resíduo | Interpretação |
|-----------------|----------------|
| **> +1,96** | Frequência observada **maior** que o esperado (associação positiva significativa) |
| **< −1,96** | Frequência observada **menor** que o esperado (associação negativa significativa) |
| Entre **−1,96 e +1,96** | Célula dentro do esperado (sem associação local significativa) |

## 🔄 ANACOR vs. ACM — Quando usar cada uma?

| Critério | ANACOR | ACM |
|----------|--------|-----|
| Número de variáveis | **2** | **3 ou mais** |
| Tabela de entrada | Contingência 2D | Matriz binária de indicadores |
| Número de dimensões | $\min(I-1, J-1)$ | $\min(J-K, \ldots)$ |
| Pacote Python | `prince.CA` | `prince.MCA` |

> **Dica:** quando um dos pares de variáveis da ANACOR tiver apenas 2 categorias (binária),  
> só há 1 dimensão — não é gerado mapa. Nesse caso, analyze os resíduos padronizados ajustados.

## ✅ Quando usar a Análise de Correspondência?

| Situação | Adequado? |
|-----------|----------|
| 2 ou mais variáveis qualitativas associadas | ✅ Sim |
| Visualizar perfis de clientes, grupos, categorias | ✅ Sim |
| Complementar um K-Means com variáveis categóricas | ✅ Sim |
| Variáveis independentes (p-valor > 0,05 no qui²) | ❌ Não |
| Variáveis numéricas contínuas | ❌ Não (use PCA) |

---
# 🧭 Parte 4 — Comparativo Geral e Fluxo de Decisão

## 📋 Resumo Comparativo das Técnicas

| | **K-Means** | **PCA** | **ANACOR / ACM** |
|--|------------|---------|------------------|
| **Tipo de dado** | Quantitativo | Quantitativo | Qualitativo |
| **Objetivo** | Agrupar observações | Reduzir dimensões | Mapear associações |
| **Pré-requisito** | Padronização (Z-Score) | Bartlett p < 0,05 | Qui² p < 0,05 |
| **Saída principal** | Rótulos de cluster | Fatores latentes + escores | Mapa perceptual |
| **Escolha de K/m** | Elbow + Silhueta | Autovalor > 1 | Inércia por dimensão |
| **Validação** | ANOVA entre clusters | % variância + comunalidades | Resíduos padronizados |
| **Combina com** | ACM (para qualitativas) | K-Means (como feature) | K-Means (incluir cluster) |

## 🔀 Fluxo de Decisão

```
Tenho dados para segmentar/resumir?
         │
         ├── São quantitativos? ────────────────────────────┬
         |                                               |
         │  Quero AGRUPAR         Quero RESUMIR         |
         │  observações?         variáveis?             |
         │       │                    │                 |
         │  K-MEANS              PCA / Fatorial         |
         │  + ANOVA              + Bartlett             |
         |                        + Autovalores          |
         └── São qualitativos? ─────────────────────┘
                    │
                    ├── 2 variáveis? → ANACOR
                    └── 3+ variáveis? → ACM
```

> **Análise Híbrida:** quando o dataset possui variáveis de ambos os tipos, as técnicas podem ser **combinadas**:  
> 1. K-Means nas quantitativas → gera `Cluster`  
> 2. ACM nas qualitativas + `Cluster` → mapa perceptual integrado  
> 3. PCA nas quantitativas + coordenadas ACM → fator híbrido

## 📚 Referências e Leituras Complementares

- **Façam, L. et al. (2017)** — *Análise Fatorial* — uso de PCA em ciências sociais aplicadas
- **Greenacre, M. (2017)** — *Correspondence Analysis in Practice* — referência clássica em ANACOR/ACM
- **James, G. et al. (2021)** — *An Introduction to Statistical Learning* — capítulo de métodos não supervisionados
- **Sklearn Documentation** — [`KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [`silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- **Prince Documentation** — [`MCA`](https://github.com/MaxHalford/prince) — implementação de ACM e ANACOR em Python
- **Factor Analyzer** — [`FactorAnalyzer`](https://factor-analyzer.readthedocs.io/) — PCA e Análise Fatorial em Python